Train YOLO11n for 30 epochs (controlled experiment) on SeaDronesSee_ODv2.

This script:
1. Preflights every mandatory input.
2. Trains yolo11n.pt for 30 epochs (patience 10).
3. Validates the best checkpoint on the val split.
4. Exports per-class metrics and compares to the 2-epoch baseline.


In [1]:
import math
import shutil
import sys
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PROJECT = Path("E:/lifeguardDron")
DATASET_YAML = PROJECT / "data_prepared" / "detection" / "SeaDronesSee_ODv2" / "dataset.yaml"
PRETRAINED_PT = PROJECT / "yolo11n.pt"
SMOKE_ARGS = PROJECT / "runs" / "detect" / "seadrones_smoke_test" / "args.yaml"
SMOKE_BEST = PROJECT / "runs" / "detect" / "seadrones_smoke_test" / "weights" / "best.pt"
BASELINE_CSV = PROJECT / "reports" / "model_evaluation" / "yolo11n_validation_metrics.csv"
RUN_NAME = "seadrones_yolo11n_30ep_v1"
RUN_DIR = PROJECT / "runs" / "detect" / RUN_NAME
NEW_METRICS_CSV = PROJECT / "reports" / "model_evaluation" / "yolo11n_30ep_v1_metrics.csv"

def fail(msg):
    print("ERROR:", msg)
    sys.exit(1)

def require(cond, msg):
    if not cond:
        fail("PRECHECK FAILED: " + msg)
    print("  OK  " + msg)

# ---------------------------------------------------------------------------
# 1. Preflight
# ---------------------------------------------------------------------------
print("=== Preflight ===")
require(PROJECT.exists(), "project root exists")
require(DATASET_YAML.exists(), "dataset.yaml exists")

with open(DATASET_YAML, encoding="utf-8") as f:
    data = yaml.safe_load(f)
print("  YAML path:", data.get("path"))
print("  train:", data.get("train"), "| val:", data.get("val"), "| nc:", data.get("nc"))

names = data.get("names")
expected = {0: "swimmer", 1: "boat", 2: "jetski", 3: "life_saving_appliances", 4: "buoy"}
require(names == expected, "class names and order match the expected 5-class mapping")

dataset_root = Path(data["path"])
for d, label in [(dataset_root / data["train"], "train images"),
                 (dataset_root / data["val"], "val images"),
                 (dataset_root / "labels" / "train", "train labels"),
                 (dataset_root / "labels" / "val", "val labels")]:
    require(d.exists() and d.is_dir(), label + " directory exists: " + str(d))

require(PRETRAINED_PT.exists(), "pretrained yolo11n.pt exists: " + str(PRETRAINED_PT))
require(torch.cuda.is_available(), "CUDA is available")
print("  GPU 0:", torch.cuda.get_device_name(0))

free = shutil.disk_usage(PROJECT).free
print("  Disk free on project drive: %.1f GB" % (free / 1e9))
require(free > 2e9, "at least 2 GB free disk space")

require(SMOKE_ARGS.exists(), "smoke-test args.yaml exists")
require(SMOKE_BEST.exists(), "smoke-test best.pt exists")
require(BASELINE_CSV.exists(), "baseline metrics CSV exists")
require(not RUN_DIR.exists(), "run directory does NOT already exist: " + str(RUN_DIR))
print("Preflight passed.\n")

# ---------------------------------------------------------------------------
# 2. Training configuration
# ---------------------------------------------------------------------------
TRAIN_CONFIG = {
    "model": str(PRETRAINED_PT),
    "data": str(DATASET_YAML),
    "epochs": 30,
    "patience": 10,
    "imgsz": 640,
    "batch": 4,
    "device": 0,
    "seed": 42,
    "deterministic": True,
    "pretrained": True,
    "optimizer": "auto",
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "mosaic": 1.0,
    "erasing": 0.4,
    "cls_remap": True,
    "project": str(PROJECT / "runs" / "detect"),
    "name": RUN_NAME,
    "exist_ok": False,
    "resume": False,
    "amp": True,
    "workers": 2,
    "val": True,
    "split": "val",
}

print("=== Training configuration ===")
for k, v in TRAIN_CONFIG.items():
    print("  %-14s = %r" % (k, v))
print()

# ---------------------------------------------------------------------------
# 3. Train
# ---------------------------------------------------------------------------
print("=== Training ===")
model = YOLO(TRAIN_CONFIG["model"])
train_kwargs = {k: v for k, v in TRAIN_CONFIG.items() if k != "model"}
model.train(**train_kwargs)

BEST_PT = RUN_DIR / "weights" / "best.pt"
print("\nTraining finished.")
print("Run directory:", RUN_DIR)
print("Best checkpoint:", BEST_PT, "->", BEST_PT.exists())

# ---------------------------------------------------------------------------
# 4. Validate
# ---------------------------------------------------------------------------
print("\n=== Validation ===")
if not BEST_PT.exists():
    fail("VALIDATION FAILED: best.pt not found: " + str(BEST_PT))

metrics = model.val(data=str(DATASET_YAML), imgsz=640, device=0, iou=0.7,
                    conf=0.001, split="val", plots=False, verbose=False)

names_list = [names[i] for i in range(len(names))]
ap_cls = list(map(int, metrics.box.ap_class_index))
per = {}
for idx, cls_id in enumerate(ap_cls):
    per[cls_id] = {
        "p": float(metrics.box.p[idx]),
        "r": float(metrics.box.r[idx]),
        "map50": float(metrics.box.map50[idx]),
        "map5095": float(metrics.box.map[idx]),
    }

print("%-26s %10s %10s %10s %10s" % ("class", "P", "R", "mAP50", "mAP50-95"))
new_metrics = {}
for i in range(len(names)):
    m = per.get(i)
    if m is None or not all(math.isfinite(v) for v in m.values()):
        print("%-26s %10s %10s %10s %10s" % (names_list[i], "n/a", "n/a", "n/a", "n/a"))
        new_metrics[i] = None
    else:
        print("%-26s %10.4f %10.4f %10.4f %10.4f" % (names_list[i], m["p"], m["r"], m["map50"], m["map5095"]))
        new_metrics[i] = m

# ---------------------------------------------------------------------------
# 5. Export + compare
# ---------------------------------------------------------------------------
print("\n=== Export + compare ===")
import csv

if NEW_METRICS_CSV.exists():
    fail("REFUSING to overwrite existing CSV: " + str(NEW_METRICS_CSV))
NEW_METRICS_CSV.parent.mkdir(parents=True, exist_ok=True)
with open(NEW_METRICS_CSV, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Class", "Precision", "Recall", "mAP50", "mAP50-95"])
    for i in range(len(names)):
        m = new_metrics.get(i)
        w.writerow([names_list[i],
                    "" if m is None else "%.6f" % m["p"],
                    "" if m is None else "%.6f" % m["r"],
                    "" if m is None else "%.6f" % m["map50"],
                    "" if m is None else "%.6f" % m["map5095"]])
print("Wrote new metrics:", NEW_METRICS_CSV)

base = {}
with open(BASELINE_CSV, encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if len(row) >= 5:
            base[row[0]] = {"map50": row[3], "map5095": row[4]}

def fnum(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

print("\n%-26s %10s %10s %10s | %10s %10s %10s" % (
    "class", "base_mAP50", "new_mAP50", "d_mAP50", "base_mAP5095", "new_mAP5095", "d_mAP5095"))
for i in range(len(names)):
    name = names_list[i]
    b = base.get(name)
    m = new_metrics.get(i)
    b50 = fnum(b["map50"]) if b else None
    b95 = fnum(b["map5095"]) if b else None
    n50 = m["map50"] if m else None
    n95 = m["map5095"] if m else None
    d50 = (n50 - b50) if (n50 is not None and b50 is not None) else None
    d95 = (n95 - b95) if (n95 is not None and b95 is not None) else None
    print("%-26s %10s %10s %10s | %10s %10s %10s" % (
        name,
        "%.4f" % b50 if b50 is not None else "n/a",
        "%.4f" % n50 if n50 is not None else "n/a",
        ("%+.4f" % d50) if d50 is not None else "n/a",
        "%.4f" % b95 if b95 is not None else "n/a",
        "%.4f" % n95 if n95 is not None else "n/a",
        ("%+.4f" % d95) if d95 is not None else "n/a"))

print("\nDone. Best checkpoint:", BEST_PT)

=== Preflight ===
  OK  project root exists
  OK  dataset.yaml exists
  YAML path: E:/lifeguardDron/data_prepared/detection/SeaDronesSee_ODv2
  train: images/train | val: images/val | nc: 5
  OK  class names and order match the expected 5-class mapping
  OK  train images directory exists: E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\images\train
  OK  val images directory exists: E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\images\val
  OK  train labels directory exists: E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\labels\train
  OK  val labels directory exists: E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\labels\val
  OK  pretrained yolo11n.pt exists: E:\lifeguardDron\yolo11n.pt
  OK  CUDA is available
  GPU 0: NVIDIA GeForce GTX 1650
  Disk free on project drive: 125.7 GB
  OK  at least 2 GB free disk space
  OK  smoke-test args.yaml exists
  OK  smoke-test best.pt exists
  OK  baseline metrics CSV exists
  OK  run directory doe

IndexError: invalid index to scalar variable.

## Dataset audit — 1: YAML, paths, and classes

Inspect dataset.yaml, resolve the train/val(/test) image and label directories, and
verify the five-class mapping. Read-only.

In [2]:
from pathlib import Path
import yaml

DATASET_YAML = Path("E:/lifeguardDron/data_prepared/detection/SeaDronesSee_ODv2/dataset.yaml")
EXPECTED = {0: "swimmer", 1: "boat", 2: "jetski", 3: "life_saving_appliances", 4: "buoy"}

print("YAML exists:", DATASET_YAML.exists())
cfg = yaml.safe_load(DATASET_YAML.read_text(encoding="utf-8"))

root = Path(cfg["path"])
print("root:", root, "->", root.exists())
print("nc:", cfg["nc"])
print("names:", cfg["names"])
print("train rel:", cfg.get("train"))
print("val rel:", cfg.get("val"))
print("test rel:", cfg.get("test", "(none)"))

assert cfg["names"] == EXPECTED, "class names/order mismatch"
print("Class mapping verified.")

# resolve split dirs (images/<split>  <->  labels/<split>)
SPLITS = {}
for key in ("train", "val", "test"):
    rel = cfg.get(key)
    if not rel:
        continue
    imgs = root / rel
    lbls = root / "labels" / Path(rel).name
    SPLITS[key] = {"images": imgs, "labels": lbls}
    print(f"{key}: images={imgs} exists={imgs.exists()}  labels={lbls} exists={lbls.exists()}")

YAML exists: True
root: E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2 -> True
nc: 5
names: {0: 'swimmer', 1: 'boat', 2: 'jetski', 3: 'life_saving_appliances', 4: 'buoy'}
train rel: images/train
val rel: images/val
test rel: (none)
Class mapping verified.
train: images=E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\images\train exists=True  labels=E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\labels\train exists=True
val: images=E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\images\val exists=True  labels=E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\labels\val exists=True


## Dataset audit — 2: image, label, and object counts

Count images and label files per split, then per-class object counts and per-class
image counts (an object count counts every box; an image count counts images that
contain at least one box of that class).

In [3]:
from collections import Counter

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(d):
    return sorted(p for p in d.glob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

def list_labels(d):
    return sorted(p for p in d.glob("*.txt"))

obj_counts = {}   # split -> Counter(class -> boxes)
img_counts = {}   # split -> Counter(class -> images containing class)
n_images = {}
n_labels = {}

for key, d in SPLITS.items():
    images = list_images(d["images"])
    labels = list_labels(d["labels"])
    n_images[key] = len(images)
    n_labels[key] = len(labels)
    oc = Counter()
    ic = Counter()
    for lbl in labels:
        classes_in_file = set()
        for line in lbl.read_text(encoding="utf-8").splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cls = int(float(parts[0]))
            oc[cls] += 1
            classes_in_file.add(cls)
        for c in classes_in_file:
            ic[c] += 1
    obj_counts[key] = oc
    img_counts[key] = ic

print("=== File counts per split ===")
for key in SPLITS:
    print(f"  {key}: images={n_images[key]}  labels={n_labels[key]}")

print("\n=== Object counts per class (boxes) ===")
print("  %-26s %10s %10s %10s" % ("class", "train", "val", "test"))
for cls in range(5):
    name = EXPECTED[cls]
    vals = [str(obj_counts[k].get(cls, 0)) if k in obj_counts else "-" for k in ("train", "val", "test")]
    print("  %-26s %10s %10s %10s" % (name, *vals))

print("\n=== Images containing each class ===")
print("  %-26s %10s %10s %10s" % ("class", "train", "val", "test"))
for cls in range(5):
    name = EXPECTED[cls]
    vals = [str(img_counts[k].get(cls, 0)) if k in img_counts else "-" for k in ("train", "val", "test")]
    print("  %-26s %10s %10s %10s" % (name, *vals))

=== File counts per split ===
  train: images=8930  labels=8930
  val: images=1547  labels=1547

=== Object counts per class (boxes) ===
  class                           train        val       test
  swimmer                         37096       6206          -
  boat                            13022       2214          -
  jetski                           2330        320          -
  life_saving_appliances            923        330          -
  buoy                             4389        560          -

=== Images containing each class ===
  class                           train        val       test
  swimmer                          6898       1286          -
  boat                             5870        915          -
  jetski                           2328        320          -
  life_saving_appliances            665        191          -
  buoy                             3722        352          -


## Dataset audit — 3: annotation validation

Check every label row for: exactly 5 numeric fields, class id 0-4, center in [0,1],
and positive width/height <= 1. Also find orphan labels, missing labels, empty label
files, and exact duplicate rows.

In [4]:
import math

def validate_row(parts):
    if len(parts) != 5:
        return "malformed (%d fields)" % len(parts)
    try:
        vals = [float(x) for x in parts]
    except ValueError:
        return "non-numeric field"
    cls, cx, cy, w, h = vals
    if not all(math.isfinite(v) for v in vals):
        return "non-finite value"
    if cls != int(cls) or not (0 <= cls <= 4):
        return "class id out of range"
    if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0):
        return "center out of [0,1]"
    if not (0.0 < w <= 1.0 and 0.0 < h <= 1.0):
        return "invalid width/height"
    return None

for key in SPLITS:
    images = list_images(SPLITS[key]["images"])
    labels = list_labels(SPLITS[key]["labels"])
    img_stems = {p.stem for p in images}
    lbl_stems = {p.stem for p in labels}

    orphan = sorted(lbl_stems - img_stems)
    missing = sorted(img_stems - lbl_stems)
    empty = []
    malformed = []
    dup_files = set()
    dup_rows = 0

    for lbl in labels:
        lines = [ln.strip() for ln in lbl.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if not lines:
            empty.append(lbl.name)
            continue
        seen = set()
        for ln in lines:
            parts = ln.split()
            err = validate_row(parts)
            if err:
                malformed.append((lbl.name, ln, err))
                continue
            key5 = tuple(round(float(x), 8) for x in parts)
            if key5 in seen:
                dup_rows += 1
                dup_files.add(lbl.name)
            seen.add(key5)

    print("=== %s ===" % key)
    print("  orphan labels (no image):", len(orphan), orphan[:5])
    print("  missing labels (image w/o label):", len(missing), missing[:5])
    print("  empty label files:", len(empty), empty[:5])
    print("  malformed rows:", len(malformed))
    for (fn, ln, err) in malformed[:5]:
        print("      %s: %r -> %s" % (fn, ln, err))
    print("  files with duplicate rows:", len(dup_files), " duplicate rows removed:", dup_rows)

=== train ===
  orphan labels (no image): 0 []
  missing labels (image w/o label): 0 []
  empty label files: 0 []
  malformed rows: 0
  files with duplicate rows: 16  duplicate rows removed: 16
=== val ===
  orphan labels (no image): 0 []
  missing labels (image w/o label): 0 []
  empty label files: 0 []
  malformed rows: 0
  files with duplicate rows: 0  duplicate rows removed: 0


## Dataset audit — 4: unreadable images, leakage, and readiness

Try to open every image with OpenCV (returns None if unreadable), check for train/val
image-name overlap, and print a readiness summary. Reading all images takes a few
minutes.

In [5]:
import cv2

# cross-split leakage
train_stems = {p.stem for p in list_images(SPLITS["train"]["images"])}
val_stems = {p.stem for p in list_images(SPLITS["val"]["images"])}
print("train/val overlapping image stems:", len(train_stems & val_stems))

# unreadable images
unreadable = {}
for key in SPLITS:
    bad = []
    for p in list_images(SPLITS[key]["images"]):
        if cv2.imread(str(p)) is None:
            bad.append(p.name)
    unreadable[key] = bad
    print("%s unreadable images: %d %s" % (key, len(bad), bad[:5]))

print("\n=== Readiness summary ===")
for key in SPLITS:
    print("  %-5s: images=%d labels=%d objects=%d unreadable=%d" % (
        key, n_images[key], n_labels[key], sum(obj_counts[key].values()), len(unreadable[key])))

total_unreadable = sum(len(v) for v in unreadable.values())
print("\nDataset READY for training." if total_unreadable == 0
      else "\nDataset has issues to review before training.")

train/val overlapping image stems: 0
train unreadable images: 0 []
val unreadable images: 0 []

=== Readiness summary ===
  train: images=8930 labels=8930 objects=57760 unreadable=0
  val  : images=1547 labels=1547 objects=9630 unreadable=0

Dataset READY for training.


## Section 1 — Environment preflight

**Purpose:** Confirm the Python environment can run YOLO11 GPU training before any
training code is written or executed.

**What it checks:** Python and Ultralytics versions, PyTorch version, CUDA
availability, the GPU name, and total VRAM reported by PyTorch.

**Expected output:** Version strings, `CUDA available: True`, and the GPU name + VRAM.

**Stop condition:** Nothing here stops the notebook; it only reports. If `torch` or
`ultralytics` fails to import, fix the environment before continuing. No package is
installed, upgraded, or uninstalled.

In [6]:
import sys
import platform
import torch
import ultralytics

print("Python:", sys.version.split()[0])
print("Platform:", platform.system(), platform.machine())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM (GB): %.1f" % (props.total_memory / 1e9))
print("Ultralytics:", ultralytics.__version__)

Python: 3.11.4
Platform: Windows AMD64
PyTorch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: NVIDIA GeForce GTX 1650
VRAM (GB): 4.3
Ultralytics: 8.4.182


## Section 2 — Dataset and checkpoint validation

**Purpose:** Confirm every input needed for training exists and resolves correctly
before the training cell is reached.

**What it checks:** the dataset YAML and its resolved train/val image+label
directories, the five-class mapping, the original pretrained `yolo11n.pt`, CUDA/GPU 0,
free disk space, and that the intended output directory does not already exist.

**Expected output:** A list of `OK` lines, then "All preflight checks passed."

**Stop condition:** Any failed check raises `SystemExit` with an actionable message and
stops the notebook before training. Nothing here starts training or modifies files.

In [8]:
from pathlib import Path
import shutil
import yaml
import torch

PROJECT = Path("E:/lifeguardDron")
DATASET_YAML = PROJECT / "data_prepared" / "detection" / "SeaDronesSee_ODv2" / "dataset.yaml"
PRETRAINED_PT = PROJECT / "yolo11n.pt"
RUN_NAME = "seadrones_yolo11n_30ep_v1"
RUN_DIR = PROJECT / "runs" / "detect" / RUN_NAME
EXPECTED_NAMES = {0: "swimmer", 1: "boat", 2: "jetski", 3: "life_saving_appliances", 4: "buoy"}

def require(cond, msg):
    if not cond:
        raise SystemExit("PRECHECK FAILED: " + msg)
    print("  OK  " + msg)

# 1. dataset YAML and resolved paths
require(DATASET_YAML.exists(), "dataset.yaml exists")
cfg = yaml.safe_load(DATASET_YAML.read_text(encoding="utf-8"))
print("  YAML path:", cfg.get("path"))
print("  train:", cfg.get("train"), "| val:", cfg.get("val"), "| nc:", cfg.get("nc"))

# 2. class names and order
names = cfg.get("names")
require(names == EXPECTED_NAMES, "class names and order match the expected 5-class mapping")

# 3. resolved split dirs (images/<split>  <->  labels/<split>)
root = Path(cfg["path"])
require(root.exists(), "dataset root exists")
for split in ("train", "val"):
    rel = cfg.get(split)
    require(bool(rel), "YAML defines the '" + split + "' path")
    imgs = root / rel
    lbls = root / "labels" / Path(rel).name
    require(imgs.exists() and imgs.is_dir(), split + " images dir exists: " + str(imgs))
    require(lbls.exists() and lbls.is_dir(), split + " labels dir exists: " + str(lbls))

# 4. original pretrained weights
require(PRETRAINED_PT.exists(), "pretrained yolo11n.pt exists: " + str(PRETRAINED_PT))
print("  pretrained size (MB): %.1f" % (PRETRAINED_PT.stat().st_size / 1e6))

# 5. CUDA / GPU 0
require(torch.cuda.is_available(), "CUDA available")
print("  GPU 0:", torch.cuda.get_device_name(0))

# 6. disk space
free = shutil.disk_usage(PROJECT).free
print("  free disk (GB): %.1f" % (free / 1e9))
require(free > 2e9, ">= 2 GB free disk space")

# 7. output directory must not already exist (no overwrite)
require(not RUN_DIR.exists(), "run dir does NOT already exist: " + str(RUN_DIR))

print("\nAll preflight checks passed.")

  OK  dataset.yaml exists
  YAML path: E:/lifeguardDron/data_prepared/detection/SeaDronesSee_ODv2
  train: images/train | val: images/val | nc: 5
  OK  class names and order match the expected 5-class mapping
  OK  dataset root exists
  OK  YAML defines the 'train' path
  OK  train images dir exists: E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\images\train
  OK  train labels dir exists: E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\labels\train
  OK  YAML defines the 'val' path
  OK  val images dir exists: E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\images\val
  OK  val labels dir exists: E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\labels\val
  OK  pretrained yolo11n.pt exists: E:\lifeguardDron\yolo11n.pt
  pretrained size (MB): 5.6
  OK  CUDA available
  GPU 0: NVIDIA GeForce GTX 1650
  free disk (GB): 125.7
  OK  >= 2 GB free disk space


SystemExit: PRECHECK FAILED: run dir does NOT already exist: E:\lifeguardDron\runs\detect\seadrones_yolo11n_30ep_v1

## Continue from existing run — 1: read-only precheck

**Purpose:** Confirm the completed checkpoint and dataset are present, and that the
target output CSV does not already exist.

**Expected output:** Three `OK` lines.

**Failure conditions:** A missing checkpoint or dataset YAML, or an existing output
CSV, raises `SystemExit` (no overwrite). Reuses `PROJECT`, `RUN_DIR`, `DATASET_YAML`
from Section 2.

In [9]:
BEST_PT = RUN_DIR / "weights" / "best.pt"
BASELINE_CSV = PROJECT / "reports" / "model_evaluation" / "yolo11n_validation_metrics.csv"
OUT_CSV = PROJECT / "reports" / "model_evaluation" / "seadrones_yolo11n_30ep_v1_metrics.csv"

for cond, msg in [
    (BEST_PT.exists(), "best checkpoint exists: " + str(BEST_PT)),
    (DATASET_YAML.exists(), "dataset YAML exists"),
    (not OUT_CSV.exists(), "output CSV does NOT already exist: " + str(OUT_CSV)),
]:
    if not cond:
        raise SystemExit("PRECHECK FAILED: " + msg)
    print("  OK  " + msg)

  OK  best checkpoint exists: E:\lifeguardDron\runs\detect\seadrones_yolo11n_30ep_v1\weights\best.pt
  OK  dataset YAML exists
  OK  output CSV does NOT already exist: E:\lifeguardDron\reports\model_evaluation\seadrones_yolo11n_30ep_v1_metrics.csv


## Continue from existing run — 2: load and validate

**Purpose:** Load the trained best checkpoint and run validation on the val split.
This does NOT call model.train().

**Expected output:** Ultralytics' val progress and summary line.

**Failure conditions:** If the checkpoint cannot be loaded, the cell raises and stops.

In [10]:
from ultralytics import YOLO

print("Loading checkpoint:", BEST_PT)
model = YOLO(str(BEST_PT))
metrics = model.val(data=str(DATASET_YAML), imgsz=640, device=0, iou=0.7,
                    conf=0.001, split="val", plots=False, verbose=False)
print("Validation complete.")

Loading checkpoint: E:\lifeguardDron\runs\detect\seadrones_yolo11n_30ep_v1\weights\best.pt
Ultralytics 8.4.182  Python-3.11.4 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
YOLO11n summary (fused): 101 layers, 2,598,023 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.2±0.0 ms, read: 968.0±672.4 MB/s, size: 704.1 KB)
val: Scanning E:\lifeguardDron\data_prepared\detection\SeaDronesSee_ODv2\labels\val.cache... 1547 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1547/1547  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 97/97 1.4it/s 1:110.2sss8
                   all       1547       9630       0.83      0.652      0.668      0.375
Speed: 1.4ms preprocess, 5.3ms inference, 0.0ms loss, 1.8ms postprocess per image
Validation complete.


## Continue from existing run — 3 (fixed): per-class metrics

`metrics.box.ap` and `metrics.box.ap50` are 1-D per-class arrays; use them directly
(no 2-D indexing). Defines `per_class`, which the comparison and export cells need.

In [14]:
import math

ap50 = metrics.box.ap50        # per-class mAP@0.5      (1-D, length nc)
ap = metrics.box.ap            # per-class mAP@0.5:0.95 (1-D, length nc)
ap_cls = list(map(int, metrics.box.ap_class_index))

names_list = [names[i] for i in range(len(names))]

per = {}
for idx, cls_id in enumerate(ap_cls):
    per[cls_id] = {
        "p": float(metrics.box.p[idx]),
        "r": float(metrics.box.r[idx]),
        "map50": float(ap50[idx]),
        "map5095": float(ap[idx]),
    }

print("=== Per-class metrics (best.pt) ===")
print("%-26s %10s %10s %10s %10s" % ("class", "P", "R", "mAP50", "mAP50-95"))
per_class = {}
for i in range(len(names)):
    m = per.get(i)
    if m is None or not all(math.isfinite(v) for v in m.values()):
        print("%-26s %10s %10s %10s %10s" % (names_list[i], "n/a", "n/a", "n/a", "n/a"))
        per_class[i] = None
    else:
        print("%-26s %10.4f %10.4f %10.4f %10.4f" % (names_list[i], m["p"], m["r"], m["map50"], m["map5095"]))
        per_class[i] = m

=== Per-class metrics (best.pt) ===
class                               P          R      mAP50   mAP50-95
swimmer                        0.7584     0.7187     0.6882     0.2677
boat                           0.9187     0.9133     0.9347     0.6401
jetski                         0.8098     0.8531     0.8716     0.5132
life_saving_appliances         0.7756     0.2303     0.2589     0.0913
buoy                           0.8864     0.5464     0.5858     0.3620


## Continue from existing run — 4: baseline comparison

**Purpose:** Inspect the baseline CSV schema, then compare baseline vs new metrics
matched by class NAME (not row order).

**Expected output:** The baseline header, a side-by-side mAP table, and a highlighted
note for life_saving_appliances.

**Failure conditions:** Missing/incompatible baseline columns are reported as `n/a`
without inventing results.

In [15]:
import csv

with open(BASELINE_CSV, encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)
    print("baseline header:", header)

base = {}
with open(BASELINE_CSV, encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if len(row) >= 5:
            base[row[0]] = {"p": row[1], "r": row[2], "map50": row[3], "map5095": row[4]}

def fnum(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

print("\n=== Baseline vs 30-epoch (matched by class name) ===")
print("%-26s %10s %10s %10s | %10s %10s %10s" % (
    "class", "base_mAP50", "new_mAP50", "d_mAP50", "base_mAP5095", "new_mAP5095", "d_mAP5095"))
for i in range(len(names)):
    name = names_list[i]
    b = base.get(name)
    m = per_class.get(i)
    b50 = fnum(b["map50"]) if b else None
    b95 = fnum(b["map5095"]) if b else None
    n50 = m["map50"] if m else None
    n95 = m["map5095"] if m else None
    d50 = (n50 - b50) if (n50 is not None and b50 is not None) else None
    d95 = (n95 - b95) if (n95 is not None and b95 is not None) else None
    print("%-26s %10s %10s %10s | %10s %10s %10s" % (
        name,
        "%.4f" % b50 if b50 is not None else "n/a",
        "%.4f" % n50 if n50 is not None else "n/a",
        ("%+.4f" % d50) if d50 is not None else "n/a",
        "%.4f" % b95 if b95 is not None else "n/a",
        "%.4f" % n95 if n95 is not None else "n/a",
        ("%+.4f" % d95) if d95 is not None else "n/a"))

print("\n=== life_saving_appliances (class 3) ===")
b = base.get("life_saving_appliances")
m = per_class.get(3)
if b is not None and m is not None:
    print("  baseline: P=%s R=%s mAP50=%s mAP50-95=%s" % (b["p"], b["r"], b["map50"], b["map5095"]))
    print("  new:      P=%.4f R=%.4f mAP50=%.4f mAP50-95=%.4f" % (m["p"], m["r"], m["map50"], m["map5095"]))
else:
    print("  Could not compare (missing baseline or new metric).")

baseline header: ['Class', 'Precision', 'Recall', 'mAP@50', 'mAP@50–95']

=== Baseline vs 30-epoch (matched by class name) ===
class                      base_mAP50  new_mAP50    d_mAP50 | base_mAP5095 new_mAP5095  d_mAP5095
swimmer                        0.5769     0.6882    +0.1113 |     0.2040     0.2677    +0.0637
boat                           0.8900     0.9347    +0.0447 |     0.5333     0.6401    +0.1068
jetski                         0.7608     0.8716    +0.1109 |     0.3842     0.5132    +0.1291
life_saving_appliances         0.0998     0.2589    +0.1590 |     0.0334     0.0913    +0.0579
buoy                           0.4624     0.5858    +0.1235 |     0.2676     0.3620    +0.0944

=== life_saving_appliances (class 3) ===
  baseline: P=1.0 R=0.0 mAP50=0.09984620810128601 mAP50-95=0.03337259027035632
  new:      P=0.7756 R=0.2303 mAP50=0.2589 mAP50-95=0.0913


## Continue from existing run — 5: export metrics

**Purpose:** Write the new per-class metrics to a NEW CSV. Never overwrites.

**Expected output:** The written CSV path.

**Failure conditions:** If the CSV already exists, the cell stops with a clear message.

In [18]:
import csv

if OUT_CSV.exists():
    raise SystemExit("REFUSING to overwrite existing CSV: " + str(OUT_CSV))

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_CSV, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Class", "Precision", "Recall", "mAP50", "mAP50-95"])
    for i in range(len(names)):
        m = per_class.get(i)
        w.writerow([
            names_list[i],
            "" if m is None else "%.6f" % m["p"],
            "" if m is None else "%.6f" % m["r"],
            "" if m is None else "%.6f" % m["map50"],
            "" if m is None else "%.6f" % m["map5095"],
        ])
print("Wrote:", OUT_CSV)

SystemExit: REFUSING to overwrite existing CSV: E:\lifeguardDron\reports\model_evaluation\seadrones_yolo11n_30ep_v1_metrics.csv

In [19]:
from pathlib import Path

print("Exists:", OUT_CSV.exists())
if OUT_CSV.exists():
    print("Size:", OUT_CSV.stat().st_size, "bytes")
    print("Content:")
    print(OUT_CSV.read_text(encoding="utf-8"))

Exists: True
Size: 39 bytes
Content:
Class,Precision,Recall,mAP50,mAP50-95



In [20]:
print("per_class exists:", "per_class" in globals())
print("names exists:", "names" in globals())
print("names_list exists:", "names_list" in globals())

if "per_class" in globals():
    print("per_class type:", type(per_class).__name__)
    print("per_class:", per_class)

if "names" in globals():
    print("names type:", type(names).__name__)
    print("names:", names)

if "names_list" in globals():
    print("names_list:", names_list)

per_class exists: True
names exists: True
names_list exists: True
per_class type: dict
per_class: {0: {'p': 0.7583972693573321, 'r': 0.7186593619078311, 'map50': 0.6881966421900723, 'map5095': 0.2677490454458782}, 1: {'p': 0.9186744359999978, 'r': 0.9132926548335403, 'map50': 0.9346532245838834, 'map5095': 0.6401114935804666}, 2: {'p': 0.8097579034127321, 'r': 0.853125, 'map50': 0.8716350372335178, 'map5095': 0.5132481231919066}, 3: {'p': 0.7756072918536079, 'r': 0.23030303030303031, 'map50': 0.25887400736802857, 'map5095': 0.09127524143035061}, 4: {'p': 0.8863563723786302, 'r': 0.5464285714285714, 'map50': 0.5858280014122507, 'map5095': 0.3619623097393301}}
names type: dict
names: {0: 'swimmer', 1: 'boat', 2: 'jetski', 3: 'life_saving_appliances', 4: 'buoy'}
names_list: ['swimmer', 'boat', 'jetski', 'life_saving_appliances', 'buoy']


## Continue from existing run — 5 (fixed): export metrics

**Purpose:** Write the validated per-class metrics to a NEW CSV, overwriting only a
header-only stub left by the earlier failed run (never a file with real data).

**What it checks:** builds 5 rows from `per_class`, refuses if an existing file has any
data rows, writes, then re-reads to validate the header, row count, and class names.

**Expected output:** "Wrote: …" then "Validated: 5 data rows, correct header and class
names."

**Failure conditions:** A non-empty existing CSV, or a validation mismatch, raises
`SystemExit` and stops.

In [21]:
import csv

HEADER = ["Class", "Precision", "Recall", "mAP50", "mAP50-95"]

names_list = [names[i] for i in range(len(names))]

rows = []
for i in range(len(names)):
    m = per_class.get(i)
    rows.append([
        names_list[i],
        "" if m is None else "%.6f" % m["p"],
        "" if m is None else "%.6f" % m["r"],
        "" if m is None else "%.6f" % m["map50"],
        "" if m is None else "%.6f" % m["map5095"],
    ])

# Overwrite only a header-only stub; refuse any file with real data.
if OUT_CSV.exists():
    with open(OUT_CSV, encoding="utf-8") as f:
        existing = list(csv.reader(f))
    data_rows = [r for r in existing if r and r[0] != "Class"]
    if data_rows:
        raise SystemExit("REFUSING to overwrite existing CSV with %d data row(s): %s"
                         % (len(data_rows), OUT_CSV))
    print("Existing CSV is a header-only stub; safe to overwrite.")

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_CSV, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(HEADER)
    w.writerows(rows)
print("Wrote:", OUT_CSV)

# Validate the written output.
with open(OUT_CSV, encoding="utf-8") as f:
    written = list(csv.reader(f))
if not written or written[0] != HEADER:
    raise SystemExit("VALIDATION FAILED: header mismatch")
data = [r for r in written[1:] if r]
if len(data) != 5:
    raise SystemExit("VALIDATION FAILED: expected 5 data rows, got %d" % len(data))
got_names = [r[0] for r in data]
if got_names != names_list:
    raise SystemExit("VALIDATION FAILED: class names mismatch: %r" % got_names)
print("Validated: 5 data rows, correct header and class names.")
print("Classes:", got_names)

Existing CSV is a header-only stub; safe to overwrite.
Wrote: E:\lifeguardDron\reports\model_evaluation\seadrones_yolo11n_30ep_v1_metrics.csv
Validated: 5 data rows, correct header and class names.
Classes: ['swimmer', 'boat', 'jetski', 'life_saving_appliances', 'buoy']
